# 01 - Improvement: average several projections

**Idea.** The paper's projection draws k random latents and keeps the closest generated embedding, so the defended model is a *random* function: two projections of the same input can land on different manifold points and give different logits. Averaging the class probabilities of R independent projections removes that variance from the prediction. An attacker's greedy search is then guided by a smoother signal, and a borderline example is less likely to flip by chance.

**What changes vs the paper:** nothing in training. Same fine-tuned model, same InfoGAN, same attacks and constraints. Only the victim averages R projections (default R=5).

**Cost, stated plainly:** each query costs about R times the projection compute of baseline TMD (the paper's TMD already takes about 2x vanilla inference). The paper's protocol counts queries, not seconds, so this does not break the protocol, but it is not free.

**What counts as beating the benchmark:** the final table marks a metric BEATS only if our number exceeds the paper's TMD number by more than the combined 95% sampling noise. It also shows the delta against our own reproduction from notebook 00, which is the cleaner comparison. Run notebook 00 first.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import torch, tmd_lab as T
T.setup_env()
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'   # must match notebook 00
N_EVAL = 1000
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run'
R = 5                 # number of averaged projections
COMBINE = 'prob'      # 'prob' = mean softmax, 'logit' = mean logits

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base')
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)

# Baseline results are cached by notebook 00, so this is free if 00 already ran.
base = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)
T.evaluate(lab, base, 'baseline', E_te, Y_te, n=N_EVAL)

In [ ]:
TAG = f'ens{R}_{COMBINE}'
ens = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K, R=R, combine=COMBINE)
T.evaluate(lab, ens, TAG, E_te, Y_te, n=N_EVAL)
display(T.compare(lab, TAG, baseline_tag='baseline'))

Try other settings by changing `R` (for example 3, 10) and re-running the last two cells. Each setting is cached under its own name, so earlier results are kept. If the gain grows with R and flattens, the variance explanation is plausible; if it does not move, it is not the bottleneck.